# Clothing Classification Pipeline with Pretrained EfficientNet

Pipeline:
1. OWL-ViT object localization and cropping
2. Data augmentation for imbalanced dataset
3. Pretrained EfficientNet-B0 for classification (jenis and warna)
4. Evaluation with Exact Match metric

## 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
import cv2
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import OwlViTProcessor, OwlViTForObjectDetection
from transformers import AutoImageProcessor, AutoModelForImageClassification
from transformers import SamModel, SamProcessor
import torchvision.transforms as transforms
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")

## 2. Configuration

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = 'train/train/'
test_dir = 'test/test/'
output_dir = 'results/'
os.makedirs(output_dir, exist_ok=True)

BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4
IMG_SIZE = 224

print(f"Device: {device}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

## 3. Load Data

In [ ]:
train_df = pd.read_csv('train.csv')

print(f"Total samples: {len(train_df)}")
print(f"\nJenis distribution:")
print(train_df['jenis'].value_counts())
print(f"\nWarna distribution:")
print(train_df['warna'].value_counts())

## 4. Initialize OWL-ViT Model

In [ ]:
owl_processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
owl_model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device)
owl_model.eval()

sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam_model = SamModel.from_pretrained("facebook/sam-vit-base").to(device)
sam_model.eval()

text_queries = [["clothing", "shirt", "dress", "garment", "apparel"]]

print("OWL-ViT and SAM models loaded")

## 5. Localization Helper Functions

In [ ]:
def load_image(image_id, image_dir):
    for ext in ['.jpg', '.png', '.jpeg']:
        image_path = os.path.join(image_dir, f"{image_id}{ext}")
        if os.path.exists(image_path):
            return Image.open(image_path).convert('RGB'), image_path
    return None, None

def enhance_image_quality(image):
    """Enhance image quality: sharpening and contrast adjustment"""
    enhancer = ImageEnhance.Sharpness(image)
    image = enhancer.enhance(1.5)
    
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(1.2)
    
    image = image.filter(ImageFilter.UnsharpMask(radius=1, percent=150, threshold=3))
    
    return image

def detect_object(image, threshold=0.1):
    inputs = owl_processor(text=text_queries, images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = owl_model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = owl_processor.post_process_object_detection(
        outputs=outputs, threshold=threshold, target_sizes=target_sizes
    )[0]
    
    return results["boxes"].cpu().numpy(), results["scores"].cpu().numpy()

def segment_with_sam(image, box):
    """Use SAM to segment object from bounding box"""
    image_np = np.array(image)
    
    input_boxes = [[[box[0], box[1], box[2], box[3]]]]
    
    inputs = sam_processor(
        image, 
        input_boxes=input_boxes,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = sam_model(**inputs)
    
    masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )[0]
    
    mask = masks[0][0].numpy()
    
    return mask

def apply_mask_and_crop(image, mask, box, padding=10):
    """Apply mask to isolate object and crop with white background"""
    image_np = np.array(image)
    height, width = image_np.shape[:2]
    
    x1, y1, x2, y2 = map(int, box)
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(width, x2 + padding)
    y2 = min(height, y2 + padding)
    
    mask_binary = (mask > 0.5).astype(np.uint8)
    
    white_bg = np.ones_like(image_np) * 255
    
    masked_image = image_np * mask_binary[:, :, np.newaxis] + white_bg * (1 - mask_binary[:, :, np.newaxis])
    
    cropped = masked_image[y1:y2, x1:x2]
    
    return Image.fromarray(cropped.astype(np.uint8))

def process_image_with_sam(image):
    """Complete pipeline: enhance -> detect -> segment -> crop"""
    enhanced = enhance_image_quality(image)
    
    boxes, scores = detect_object(enhanced)
    
    if len(boxes) == 0:
        return enhanced
    
    mask = segment_with_sam(enhanced, boxes[0])
    
    result = apply_mask_and_crop(enhanced, mask, boxes[0])
    
    return result

print("SAM segmentation and enhancement functions defined")

## 6. Preprocess and Crop All Images

In [ ]:
cropped_images = []
labels_jenis = []
labels_warna = []
valid_indices = []

print("Processing images with SAM segmentation and enhancement...")
for idx, row in train_df.iterrows():
    image, _ = load_image(row['id'], train_dir)
    
    if image is not None:
        try:
            processed = process_image_with_sam(image)
            
            cropped_images.append(processed)
            labels_jenis.append(row['jenis'])
            labels_warna.append(row['warna'])
            valid_indices.append(idx)
        except Exception as e:
            print(f"Error processing image {row['id']}: {str(e)}")
            continue
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(train_df)}")

print(f"\nTotal processed images: {len(cropped_images)}")

## 7. Create Label Mappings

In [ ]:
unique_jenis = sorted(set(labels_jenis))
unique_warna = sorted(set(labels_warna))

jenis_to_idx = {label: idx for idx, label in enumerate(unique_jenis)}
idx_to_jenis = {idx: label for label, idx in jenis_to_idx.items()}

warna_to_idx = {label: idx for idx, label in enumerate(unique_warna)}
idx_to_warna = {idx: label for label, idx in warna_to_idx.items()}

labels_jenis_idx = [jenis_to_idx[label] for label in labels_jenis]
labels_warna_idx = [warna_to_idx[label] for label in labels_warna]

num_jenis_classes = len(unique_jenis)
num_warna_classes = len(unique_warna)

print(f"Number of Jenis classes: {num_jenis_classes}")
print(f"Number of Warna classes: {num_warna_classes}")

## 8. Data Augmentation

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Augmentation transforms defined")

## 9. Custom Dataset Class

In [ ]:
class ClothingDataset(Dataset):
    def __init__(self, images, labels_jenis, labels_warna, transform=None):
        self.images = images
        self.labels_jenis = labels_jenis
        self.labels_warna = labels_warna
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label_jenis = self.labels_jenis[idx]
        label_warna = self.labels_warna[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label_jenis, label_warna

print("Dataset class defined")

## 10. Train-Validation Split

In [ ]:
indices = np.arange(len(cropped_images))
train_indices, val_indices = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels_jenis_idx
)

train_images = [cropped_images[i] for i in train_indices]
train_jenis = [labels_jenis_idx[i] for i in train_indices]
train_warna = [labels_warna_idx[i] for i in train_indices]

val_images = [cropped_images[i] for i in val_indices]
val_jenis = [labels_jenis_idx[i] for i in val_indices]
val_warna = [labels_warna_idx[i] for i in val_indices]

train_dataset = ClothingDataset(train_images, train_jenis, train_warna, train_transform)
val_dataset = ClothingDataset(val_images, val_jenis, val_warna, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

## 11. Visualize SAM Segmentation Results

In [ ]:
num_samples = min(4, len(cropped_images))

if num_samples > 0:
    fig = plt.figure(figsize=(20, 12))
    
    sample_indices = np.random.choice(len(cropped_images), num_samples, replace=False)
    
    for idx in range(num_samples):
        img_idx = sample_indices[idx]
        image_id = train_df.iloc[valid_indices[img_idx]]['id']
        jenis_label = labels_jenis[img_idx]
        warna_label = labels_warna[img_idx]
        
        original_img, _ = load_image(image_id, train_dir)
        
        if original_img is not None:
            enhanced_img = enhance_image_quality(original_img)
            boxes, scores = detect_object(enhanced_img)
            
            if len(boxes) > 0:
                mask = segment_with_sam(enhanced_img, boxes[0])
                final_img = apply_mask_and_crop(enhanced_img, mask, boxes[0])
                
                ax1 = plt.subplot(num_samples, 4, idx*4 + 1)
                ax1.imshow(original_img)
                ax1.set_title(f'Original\nID: {image_id}', fontsize=10)
                ax1.axis('off')
                
                ax2 = plt.subplot(num_samples, 4, idx*4 + 2)
                ax2.imshow(enhanced_img)
                box = boxes[0]
                rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], 
                                    fill=False, color='red', linewidth=2)
                ax2.add_patch(rect)
                ax2.set_title(f'Enhanced + Detection\nScore: {scores[0]:.3f}', fontsize=10)
                ax2.axis('off')
                
                ax3 = plt.subplot(num_samples, 4, idx*4 + 3)
                mask_display = np.repeat(mask[:, :, np.newaxis], 3, axis=2)
                ax3.imshow(mask_display, cmap='gray')
                ax3.set_title('SAM Mask', fontsize=10)
                ax3.axis('off')
                
                ax4 = plt.subplot(num_samples, 4, idx*4 + 4)
                ax4.imshow(final_img)
                ax4.set_title(f'Final Result\nJenis: {jenis_label}\nWarna: {warna_label}', fontsize=10)
                ax4.axis('off')
    
    plt.tight_layout()
    plt.suptitle('SAM Segmentation Pipeline: Original → Enhanced → Mask → Final', 
                fontsize=16, y=0.995, fontweight='bold')
    plt.savefig(os.path.join(output_dir, 'sam_segmentation_pipeline.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nSAM Segmentation Pipeline Visualization:")
    print("Column 1: Original image")
    print("Column 2: Enhanced image with bounding box (red)")
    print("Column 3: SAM segmentation mask (white = object, black = background)")
    print("Column 4: Final result (masked + cropped + white background)")
    print("\nBenefits:")
    print("  - Precise object boundaries with pixel-level mask")
    print("  - Background completely removed (white background)")
    print("  - Enhanced sharpness and contrast for low-quality images")
    print("  - Better separation from similar-colored backgrounds")
    print(f"  - Successfully processed {len(cropped_images)} images")

## 12. Compare Enhancement Results

In [ ]:
num_comparison = min(3, len(cropped_images))

if num_comparison > 0:
    fig, axes = plt.subplots(num_comparison, 2, figsize=(12, 5*num_comparison))
    if num_comparison == 1:
        axes = axes.reshape(1, -1)
    
    sample_indices = np.random.choice(len(cropped_images), num_comparison, replace=False)
    
    for idx in range(num_comparison):
        img_idx = sample_indices[idx]
        image_id = train_df.iloc[valid_indices[img_idx]]['id']
        jenis_label = labels_jenis[img_idx]
        warna_label = labels_warna[img_idx]
        
        original_img, _ = load_image(image_id, train_dir)
        processed_img = cropped_images[img_idx]
        
        if original_img is not None:
            axes[idx, 0].imshow(original_img)
            axes[idx, 0].set_title(f'Before SAM\nID: {image_id}\n(Original with potential background noise)', 
                                  fontsize=10, color='red')
            axes[idx, 0].axis('off')
            
            axes[idx, 1].imshow(processed_img)
            axes[idx, 1].set_title(f'After SAM\nJenis: {jenis_label} | Warna: {warna_label}\n(Clean object with white background)', 
                                  fontsize=10, color='green')
            axes[idx, 1].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Before vs After SAM Segmentation', fontsize=14, y=1.0, fontweight='bold')
    plt.savefig(os.path.join(output_dir, 'sam_before_after_comparison.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nKey Improvements with SAM:")
    print("Before (Left): Original images may have:")
    print("  - Complex backgrounds")
    print("  - Similar color between clothing and background")
    print("  - Low quality or blurry regions")
    print("  - Multiple objects in frame")
    print("\nAfter (Right): SAM processed images have:")
    print("  - Clean white background")
    print("  - Only the clothing object visible")
    print("  - Enhanced sharpness and contrast")
    print("  - Precise object boundaries")
    print("  - Better input quality for EfficientNet model")

## 13. Visualize Mask Quality

In [ ]:
num_mask_samples = min(3, len(cropped_images))

if num_mask_samples > 0:
    fig, axes = plt.subplots(num_mask_samples, 3, figsize=(15, 5*num_mask_samples))
    if num_mask_samples == 1:
        axes = axes.reshape(1, -1)
    
    sample_indices = np.random.choice(len(cropped_images), num_mask_samples, replace=False)
    
    for idx in range(num_mask_samples):
        img_idx = sample_indices[idx]
        image_id = train_df.iloc[valid_indices[img_idx]]['id']
        
        original_img, _ = load_image(image_id, train_dir)
        
        if original_img is not None:
            enhanced_img = enhance_image_quality(original_img)
            boxes, scores = detect_object(enhanced_img)
            
            if len(boxes) > 0:
                mask = segment_with_sam(enhanced_img, boxes[0])
                
                axes[idx, 0].imshow(enhanced_img)
                axes[idx, 0].set_title(f'Enhanced Image\nID: {image_id}', fontsize=10)
                axes[idx, 0].axis('off')
                
                axes[idx, 1].imshow(mask, cmap='hot')
                axes[idx, 1].set_title('SAM Mask\n(Confidence Heatmap)', fontsize=10)
                axes[idx, 1].axis('off')
                
                overlay = np.array(enhanced_img).copy()
                mask_colored = (mask[:, :, np.newaxis] * np.array([0, 255, 0])).astype(np.uint8)
                overlay = cv2.addWeighted(overlay, 0.7, mask_colored, 0.3, 0)
                
                axes[idx, 2].imshow(overlay)
                axes[idx, 2].set_title('Overlay Visualization\n(Green = Object Mask)', fontsize=10)
                axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.suptitle('SAM Mask Quality Visualization', fontsize=14, y=1.0, fontweight='bold')
    plt.savefig(os.path.join(output_dir, 'sam_mask_quality.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nMask Quality Analysis:")
    print("Column 1: Enhanced input image")
    print("Column 2: SAM confidence heatmap (warmer colors = higher confidence)")
    print("Column 3: Overlay showing exact object boundaries (green)")
    print("\nSAM provides:")
    print("  - High-precision segmentation masks")
    print("  - Accurate edge detection")
    print("  - Robust to clothing-background color similarity")
    print("  - Handles complex shapes and patterns")

## 11. Compute Class Weights

In [ ]:
jenis_weights = compute_class_weight('balanced', classes=np.unique(train_jenis), y=train_jenis)
warna_weights = compute_class_weight('balanced', classes=np.unique(train_warna), y=train_warna)

jenis_weights = torch.FloatTensor(jenis_weights).to(device)
warna_weights = torch.FloatTensor(warna_weights).to(device)

print("Class weights computed")

## 12. Define Multi-Task EfficientNet Model

In [ ]:
class MultiTaskEfficientNet(nn.Module):
    def __init__(self, num_jenis_classes, num_warna_classes):
        super(MultiTaskEfficientNet, self).__init__()
        
        self.backbone = AutoModelForImageClassification.from_pretrained(
            "google/efficientnet-b0",
            num_labels=1000,
            ignore_mismatched_sizes=False
        )
        
        # EfficientNet-B0 has 1280 features before the classifier
        hidden_size = 1280
        
        # Replace the original classifier with identity
        self.backbone.classifier = nn.Identity()
        
        self.jenis_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_jenis_classes)
        )
        
        self.warna_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_warna_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x).logits
        
        jenis_output = self.jenis_head(features)
        warna_output = self.warna_head(features)
        
        return jenis_output, warna_output

model = MultiTaskEfficientNet(num_jenis_classes, num_warna_classes).to(device)

print("Multi-task EfficientNet model initialized")

## 13. Training Setup

In [ ]:
criterion_jenis = nn.CrossEntropyLoss(weight=jenis_weights)
criterion_warna = nn.CrossEntropyLoss(weight=warna_weights)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print("Training setup complete")

## 14. Training Loop

In [ ]:
train_losses = []
val_losses = []
val_exact_matches = []
best_exact_match = 0.0

print("Starting training...\n")

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0.0
    
    for images, jenis_labels, warna_labels in train_loader:
        images = images.to(device)
        jenis_labels = jenis_labels.to(device)
        warna_labels = warna_labels.to(device)
        
        optimizer.zero_grad()
        
        jenis_outputs, warna_outputs = model(images)
        
        loss_jenis = criterion_jenis(jenis_outputs, jenis_labels)
        loss_warna = criterion_warna(warna_outputs, warna_labels)
        loss = loss_jenis + loss_warna
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    model.eval()
    val_loss = 0.0
    correct_jenis = 0
    correct_warna = 0
    correct_both = 0
    total = 0
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in val_loader:
            images = images.to(device)
            jenis_labels = jenis_labels.to(device)
            warna_labels = warna_labels.to(device)
            
            jenis_outputs, warna_outputs = model(images)
            
            loss_jenis = criterion_jenis(jenis_outputs, jenis_labels)
            loss_warna = criterion_warna(warna_outputs, warna_labels)
            loss = loss_jenis + loss_warna
            
            val_loss += loss.item()
            
            _, jenis_pred = torch.max(jenis_outputs, 1)
            _, warna_pred = torch.max(warna_outputs, 1)
            
            correct_jenis += (jenis_pred == jenis_labels).sum().item()
            correct_warna += (warna_pred == warna_labels).sum().item()
            correct_both += ((jenis_pred == jenis_labels) & (warna_pred == warna_labels)).sum().item()
            total += jenis_labels.size(0)
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    jenis_acc = 100 * correct_jenis / total
    warna_acc = 100 * correct_warna / total
    exact_match = 100 * correct_both / total
    val_exact_matches.append(exact_match)
    
    scheduler.step()
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}:")
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Jenis Acc: {jenis_acc:.2f}% | Warna Acc: {warna_acc:.2f}%")
    print(f"  Exact Match: {exact_match:.2f}%")
    
    if exact_match > best_exact_match:
        best_exact_match = exact_match
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"  Model saved (Best Exact Match: {best_exact_match:.2f}%)")
    print()

print(f"Training completed. Best Exact Match: {best_exact_match:.2f}%")

## 15. Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(val_exact_matches, label='Exact Match', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Exact Match (%)')
ax2.set_title('Validation Exact Match Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'training_history.png'), dpi=300, bbox_inches='tight')
plt.show()

## 16. Load Best Model and Evaluate

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_jenis_true = []
all_jenis_pred = []
all_warna_true = []
all_warna_pred = []

with torch.no_grad():
    for images, jenis_labels, warna_labels in val_loader:
        images = images.to(device)
        
        jenis_outputs, warna_outputs = model(images)
        
        _, jenis_pred = torch.max(jenis_outputs, 1)
        _, warna_pred = torch.max(warna_outputs, 1)
        
        all_jenis_true.extend(jenis_labels.cpu().numpy())
        all_jenis_pred.extend(jenis_pred.cpu().numpy())
        all_warna_true.extend(warna_labels.cpu().numpy())
        all_warna_pred.extend(warna_pred.cpu().numpy())

jenis_acc = accuracy_score(all_jenis_true, all_jenis_pred)
warna_acc = accuracy_score(all_warna_true, all_warna_pred)

exact_match = sum([1 for j_t, j_p, w_t, w_p in zip(all_jenis_true, all_jenis_pred, all_warna_true, all_warna_pred) 
                   if j_t == j_p and w_t == w_p]) / len(all_jenis_true)

print("Final Evaluation Results:")
print(f"Jenis Accuracy: {jenis_acc*100:.2f}%")
print(f"Warna Accuracy: {warna_acc*100:.2f}%")
print(f"Exact Match Accuracy: {exact_match*100:.2f}%")

## 17. Classification Report for Jenis

In [ ]:
print("Classification Report - Jenis:")
print(classification_report(all_jenis_true, all_jenis_pred, 
                          target_names=[str(idx_to_jenis[i]) for i in range(num_jenis_classes)]))

## 18. Classification Report for Warna

In [ ]:
print("Classification Report - Warna:")
print(classification_report(all_warna_true, all_warna_pred,
                          target_names=[str(idx_to_warna[i]) for i in range(num_warna_classes)]))

## 19. Confusion Matrix for Jenis

In [ ]:
cm_jenis = confusion_matrix(all_jenis_true, all_jenis_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Blues',
            xticklabels=[str(idx_to_jenis[i]) for i in range(num_jenis_classes)],
            yticklabels=[str(idx_to_jenis[i]) for i in range(num_jenis_classes)])
plt.title('Confusion Matrix - Jenis Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_jenis.png'), dpi=300, bbox_inches='tight')
plt.show()

## 20. Confusion Matrix for Warna

In [ ]:
cm_warna = confusion_matrix(all_warna_true, all_warna_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_warna, annot=True, fmt='d', cmap='Greens',
            xticklabels=[str(idx_to_warna[i]) for i in range(num_warna_classes)],
            yticklabels=[str(idx_to_warna[i]) for i in range(num_warna_classes)])
plt.title('Confusion Matrix - Warna Classification')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_warna.png'), dpi=300, bbox_inches='tight')
plt.show()

## 21. Analyze Misclassified Samples

In [ ]:
misclassified_indices = []
misclassified_info = []

for i in range(len(all_jenis_true)):
    jenis_true = all_jenis_true[i]
    jenis_pred = all_jenis_pred[i]
    warna_true = all_warna_true[i]
    warna_pred = all_warna_pred[i]
    
    if jenis_true != jenis_pred or warna_true != warna_pred:
        misclassified_indices.append(val_indices[i])
        misclassified_info.append({
            'index': val_indices[i],
            'jenis_true': str(idx_to_jenis[jenis_true]),
            'jenis_pred': str(idx_to_jenis[jenis_pred]),
            'warna_true': str(idx_to_warna[warna_true]),
            'warna_pred': str(idx_to_warna[warna_pred]),
            'jenis_correct': jenis_true == jenis_pred,
            'warna_correct': warna_true == warna_pred
        })

print(f"Total misclassified samples: {len(misclassified_indices)} / {len(all_jenis_true)}")
print(f"Misclassification rate: {len(misclassified_indices) / len(all_jenis_true) * 100:.2f}%")

jenis_only_wrong = sum(1 for info in misclassified_info if not info['jenis_correct'] and info['warna_correct'])
warna_only_wrong = sum(1 for info in misclassified_info if info['jenis_correct'] and not info['warna_correct'])
both_wrong = sum(1 for info in misclassified_info if not info['jenis_correct'] and not info['warna_correct'])

print(f"\nError breakdown:")
print(f"  Jenis only wrong: {jenis_only_wrong}")
print(f"  Warna only wrong: {warna_only_wrong}")
print(f"  Both wrong: {both_wrong}")

## 22. Visualize Misclassified Samples

In [ ]:
num_samples = min(12, len(misclassified_indices))

if num_samples > 0:
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    
    sample_indices = np.random.choice(len(misclassified_indices), num_samples, replace=False)
    
    for idx, ax in enumerate(axes[:num_samples]):
        info = misclassified_info[sample_indices[idx]]
        image_idx = info['index']
        image_id = train_df.iloc[image_idx]['id']
        
        image, _ = load_image(image_id, train_dir)
        
        if image is not None:
            try:
                processed = process_image_with_sam(image)
                ax.imshow(processed)
            except:
                ax.text(0.5, 0.5, 'Processing Error', ha='center', va='center')
                ax.axis('off')
                continue
            
            title = f"ID: {image_id}\n"
            title += f"Jenis: {info['jenis_true']} -> {info['jenis_pred']}"
            if not info['jenis_correct']:
                title += " (X)"
            title += f"\nWarna: {info['warna_true']} -> {info['warna_pred']}"
            if not info['warna_correct']:
                title += " (X)"
            
            color = 'red' if not info['jenis_correct'] and not info['warna_correct'] else 'orange'
            ax.set_title(title, fontsize=9, color=color)
        else:
            ax.text(0.5, 0.5, f'Image {image_id}\nNot Found', ha='center', va='center')
        
        ax.axis('off')
    
    for idx in range(num_samples, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.suptitle('Misclassified Samples (True -> Predicted)', fontsize=14, y=0.995)
    plt.savefig(os.path.join(output_dir, 'misclassified_samples.png'), dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No misclassified samples found.")

## 23. Detailed Error Analysis by Category

In [ ]:
from collections import Counter

print("Most Confused Jenis Pairs:")
jenis_confusions = []
for info in misclassified_info:
    if not info['jenis_correct']:
        jenis_confusions.append((info['jenis_true'], info['jenis_pred']))

if jenis_confusions:
    jenis_counter = Counter(jenis_confusions)
    for (true_label, pred_label), count in jenis_counter.most_common(10):
        print(f"  {true_label} -> {pred_label}: {count} times")
else:
    print("  No jenis misclassifications")

print("\nMost Confused Warna Pairs:")
warna_confusions = []
for info in misclassified_info:
    if not info['warna_correct']:
        warna_confusions.append((info['warna_true'], info['warna_pred']))

if warna_confusions:
    warna_counter = Counter(warna_confusions)
    for (true_label, pred_label), count in warna_counter.most_common(10):
        print(f"  {true_label} -> {pred_label}: {count} times")
else:
    print("  No warna misclassifications")

print("\nError rate by Jenis:")
jenis_errors = {}
for info in misclassified_info:
    if not info['jenis_correct']:
        true_label = info['jenis_true']
        jenis_errors[true_label] = jenis_errors.get(true_label, 0) + 1

jenis_totals = Counter([str(idx_to_jenis[label]) for label in all_jenis_true])
for label in sorted(jenis_totals.keys()):
    errors = jenis_errors.get(label, 0)
    total = jenis_totals[label]
    error_rate = errors / total * 100 if total > 0 else 0
    print(f"  {label}: {errors}/{total} ({error_rate:.1f}%)")

print("\nError rate by Warna:")
warna_errors = {}
for info in misclassified_info:
    if not info['warna_correct']:
        true_label = info['warna_true']
        warna_errors[true_label] = warna_errors.get(true_label, 0) + 1

warna_totals = Counter([str(idx_to_warna[label]) for label in all_warna_true])
for label in sorted(warna_totals.keys()):
    errors = warna_errors.get(label, 0)
    total = warna_totals[label]
    error_rate = errors / total * 100 if total > 0 else 0
    print(f"  {label}: {errors}/{total} ({error_rate:.1f}%)")

## 21. Test Dataset Prediction

In [ ]:
submission_df = pd.read_csv('sample_submission.csv')

print(f"Test samples: {len(submission_df)}")

## 22. Generate Test Predictions

In [ ]:
def predict_test_image(image_id):
    image, _ = load_image(image_id, test_dir)
    if image is None:
        return 0, 0
    
    try:
        processed = process_image_with_sam(image)
        
        image_tensor = val_transform(processed).unsqueeze(0).to(device)
        
        with torch.no_grad():
            jenis_output, warna_output = model(image_tensor)
            jenis_pred = torch.argmax(jenis_output, dim=1).item()
            warna_pred = torch.argmax(warna_output, dim=1).item()
        
        return jenis_pred, warna_pred
    except:
        return 0, 0

print("Generating test predictions...")

predictions_jenis = []
predictions_warna = []

for idx, row in submission_df.iterrows():
    image_id = row['id']
    jenis_pred, warna_pred = predict_test_image(image_id)
    
    predictions_jenis.append(jenis_pred)
    predictions_warna.append(warna_pred)
    
    if (idx + 1) % 50 == 0:
        print(f"Processed {idx + 1}/{len(submission_df)}")

print("Prediction completed")

## 23. Create Submission File

In [ ]:
submission_df['jenis'] = predictions_jenis
submission_df['warna'] = predictions_warna

submission_df.to_csv('submission.csv', index=False)

print("Submission file saved: submission.csv")
print("\nSubmission preview:")
print(submission_df.head(10))

print("\nPrediction distribution:")
print("Jenis:")
print(pd.Series(predictions_jenis).value_counts().sort_index())
print("\nWarna:")
print(pd.Series(predictions_warna).value_counts().sort_index())

## 24. Visualize Sample Predictions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

sample_indices = np.random.choice(len(submission_df), 6, replace=False)

for idx, ax in enumerate(axes):
    sample_idx = sample_indices[idx]
    image_id = submission_df.iloc[sample_idx]['id']
    pred_jenis = submission_df.iloc[sample_idx]['jenis']
    pred_warna = submission_df.iloc[sample_idx]['warna']
    
    image, _ = load_image(image_id, test_dir)
    
    if image is not None:
        try:
            processed = process_image_with_sam(image)
            ax.imshow(processed)
        except:
            ax.text(0.5, 0.5, 'Processing Error', ha='center', va='center')
            ax.axis('off')
            continue
        
        jenis_name = str(idx_to_jenis.get(pred_jenis, "Unknown"))
        warna_name = str(idx_to_warna.get(pred_warna, "Unknown"))
        
        ax.set_title(f'ID: {image_id}\nJenis: {jenis_name}\nWarna: {warna_name}', fontsize=10)
    else:
        ax.text(0.5, 0.5, f'Image {image_id}\nNot Found', ha='center', va='center')
    
    ax.axis('off')

plt.tight_layout()
plt.suptitle('Sample Test Predictions', fontsize=14, y=1.01)
plt.savefig(os.path.join(output_dir, 'sample_predictions.png'), dpi=300, bbox_inches='tight')
plt.show()